In [1]:
import pandas as pd
from pathlib import Path

import os

PROJECT_ROOT = Path(
    os.getenv("PROJECT_ROOT", Path.cwd())
).expanduser().resolve()


JUDGE_PATH = (
    PROJECT_ROOT


    / "llm_judge_results_answers_log_new_dataset_short5_rows_1_to_24.csv"
)


CSV_PATH_NEW =  Path(os.getenv("ANSWERS_LOG_PATH", str(JUDGE_PATH))).expanduser().resolve()
df_new = pd.read_csv(CSV_PATH_NEW)





In [2]:
counts = (
    df_new.groupby(["script", "correctness_error_severity"])
      .size()
      .unstack(fill_value=0)
)
print(counts)


correctness_error_severity                  high  low  medium
script                                                       
LLMGraph_Hybrid_Community_Retriever            1    2       0
LLMGraph_Hybrid_Retriever                      1    4       1
LLMGraph_Vector_KG_Retriever                   1    2       0
LlamaIndex_Vector_KG_Retriever                 0    2       1
SimpleKG_Hybrid_Community_Retriever_Rerank     0    2       1
SimpleKG_Hybrid_Retriever_Rerank               1    2       0
SimpleKG_Vector_KG_Retriever                   1    2       0


In [3]:
counts = (
    df_new.groupby(["script", "correctness_category"])
      .size()
      .unstack(fill_value=0)
)
counts = counts.reindex(columns=["TP", "FP", "FN"], fill_value=0)
print(counts)




correctness_category                        TP  FP  FN
script                                                
LLMGraph_Hybrid_Community_Retriever          1   1   0
LLMGraph_Hybrid_Retriever                    0   1   0
LLMGraph_Vector_KG_Retriever                 0   1   0
LlamaIndex_Vector_KG_Retriever               0   0   1
SimpleKG_Hybrid_Community_Retriever_Rerank   1   0   0
SimpleKG_Hybrid_Retriever_Rerank             1   1   0
SimpleKG_Vector_KG_Retriever                 0   1   0


In [4]:


cols = ["script",  "faithfulness_1to5", "answer_relevance_1to5","completeness_1to5","correctness_coverage_1to5" ,"helpfulness_final_1to5" ]
df_new_scores = df_new[cols].copy()



summary = (
    df_new_scores.groupby("script", as_index=False)
      .agg(
          Faith=("faithfulness_1to5", "mean"),
          Ans_Rel=("answer_relevance_1to5", "mean"),
          Cont_Rel=("completeness_1to5", "mean"),
          Corr =("correctness_coverage_1to5", "mean"),  
              Help =("helpfulness_final_1to5", "mean"),  
          
        
          N=("script", "size"),
      )
)


summary_no_n = summary.drop(columns=["N"])


summary_no_n = summary_no_n.set_index("script").round(2)

print(summary_no_n.to_string())


                                            Faith  Ans_Rel  Cont_Rel  Corr  Help
script                                                                          
LLMGraph_Hybrid_Community_Retriever          3.00     3.33      1.67  3.33  4.00
LLMGraph_Hybrid_Retriever                    3.67     3.17      2.50  2.67  4.17
LLMGraph_Vector_KG_Retriever                 5.00     3.33      2.00  1.67  4.67
LlamaIndex_Vector_KG_Retriever               4.33     3.33      1.67  2.00  4.00
SimpleKG_Hybrid_Community_Retriever_Rerank   3.00     3.00      2.67  3.00  4.00
SimpleKG_Hybrid_Retriever_Rerank             3.33     3.33      2.33  2.33  4.00
SimpleKG_Vector_KG_Retriever                 4.00     3.33      1.33  2.33  4.67


In [5]:
# -------------------------------
# 1) Spalten definieren
# -------------------------------
cols = [
    "script",
    "helpfulness_raw_1to5",
    "helpfulness_raw_score",
    "helpfulness_final_score",
    "helpfulness_final_1to5",
    "helpfulness_justification",
]

# -------------------------------
# 2) Relevante Daten auswählen
# -------------------------------
df_new_scores = df_new[cols].copy()

# -------------------------------
# 3) Mittelwerte pro Script berechnen
# -------------------------------
summary = (
    df_new_scores
        .groupby("script")
        .mean(numeric_only=True)
        .round(2)
)

# -------------------------------
# 4) (Optional) Anzahl Einträge pro Script
# -------------------------------
summary["N"] = df_new_scores.groupby("script").size()

# -------------------------------
# 5) Ausgabe
# -------------------------------
print(summary.to_string())


                                            helpfulness_raw_1to5  helpfulness_raw_score  helpfulness_final_score  helpfulness_final_1to5  N
script                                                                                                                                     
LLMGraph_Hybrid_Community_Retriever                         5.00                   1.00                     0.76                    4.00  3
LLMGraph_Hybrid_Retriever                                   5.00                   1.00                     0.79                    4.17  6
LLMGraph_Vector_KG_Retriever                                4.67                   0.92                     0.90                    4.67  3
LlamaIndex_Vector_KG_Retriever                              4.33                   0.83                     0.77                    4.00  3
SimpleKG_Hybrid_Community_Retriever_Rerank                  5.00                   1.00                     0.72                    4.00  3
SimpleKG_Hybrid_Retr

In [ ]:


cols = ["script", "faithfulness_score", "faithfulness_1to5", "answer_relevance_score", "answer_relevance_1to5", "completeness_score" ,"completeness_1to5","correctness_coverage" ]
df_new_scores = df_new[cols].copy()



summary = (
    df_new_scores.groupby("script", as_index=False)
      .agg(
          Faith=("faithfulness_score", "mean"),
          Ans_Rel=("answer_relevance_score", "mean"),
          Cont_Rel=("completeness_score", "mean"),
          Faith2=("faithfulness_1to5", "mean"),
          Ans_Rel2=("answer_relevance_1to5", "mean"),
          Cont_Rel2=("completeness_1to5", "mean"),
          Corr =("correctness_coverage", "mean"),
          N=("script", "size"),
      )
)



cat_table = cat_counts.pivot_table(
    index=["query_type", "script"],
    columns="correctness_category",
    values="count",
    fill_value=0,
    aggfunc="sum"
).sort_index()

display(cat_table)


correctness_category                      fn  fp  tp
query_type     script                               
disambiguation KG_PGRetriever_LlamaIndex   1   6  15
               ONLY_RAG_Advanced          14   6   2
factual        KG_PGRetriever_LlamaIndex   7   7  15
               ONLY_RAG_Advanced          27   0   1
multi_hop      KG_PGRetriever_LlamaIndex   0  12  10
               ONLY_RAG_Advanced          14   8   0
reasoning      KG_PGRetriever_LlamaIndex   2  18   9
               ONLY_RAG_Advanced          16  12   1
relational     KG_PGRetriever_LlamaIndex   3  10  13
               ONLY_RAG_Advanced          24   0   2
summary        KG_PGRetriever_LlamaIndex   8   9   6
               ONLY_RAG_Advanced          13   9   1